# Create graph figure

### Import libraries

In [1]:
import igraph
import pandas as pd
from collections import Counter

### Import data

In [2]:
df = pd.read_csv('../raw/databases_rna_sugarcane.csv')

In [3]:
# Create a mapping from time ontology IDs to human-readable names
# Used the 
# Mapping to a Units of measurement ontology (UO) for time/developmental stages
# This dictionary maps general time units (like months, days, hours) to their corresponding terms in the Units of measurement ontology (UO),
# https://www.ebi.ac.uk/ols4/ontologies/om
time2UO = {
    "UO_0000035": "months",
    "UO_0000033": "days",
    "UO_0000032": "hours",
}

# Mapping to Plant Ontology (PO) for developmental stages
# This dictionary maps specific developmental stage descriptions to their corresponding terms in the Plant Ontology (PO),
# https://bioportal.bioontology.org/ontologies/PO
time2PO = {
    "PO:0008037": "Seedling",
    "PO:0007089": "Elongation",
    "PO:0007134": "Vegetative",
    "PO:0007520": "Rooting",
    "PO:0007073": "Tillering",
    "PO:0007130": "Reproductive",
}
# Create a mapping from PECO IDs to human-readable names
# Used the Plant Experimental Conditions Ontology (PECO) database to find the corresponding names for each PECO ID
# https://bioportal.bioontology.org/ontologies/PECO
peco2name = {
    "PECO:0001062": "Control",
    " PECO:0001062": "Control",
    "PECO:0007404": "Drought",
    "PECO:0007241": "Plant Nutrient",
    "PECO:0007357": "Biotic Plant",
    "PECO:0007189": "Chemical",
    "PECO:0007191": "Abiotic Plant",
    "PECO:0007174": "Cold Temperature",
    "PECO:0007357,PECO:0007404": "Biotic + Drought",
    "PECO:0007333": "Insect Plant",
    "PECO:0007050": "Soil Texture",
    "PECO:0007078": "Light Quantity",
    "PECO:0007189,PECO:0007165": "Chemical + Plant Growth Hormone",
    "PECO:0007189,PECO:0007404": "Chemical + Drought",
    "PECO:0007185": "Salt",
    "PECO:0001037": "Oxidative Stress",
    "PECO:0007173": "High Temperature",
}
# Create a mapping from PO IDs to human-readable names
# Used the Plant Ontology (PO) database to find the corresponding names for each PO ID
# https://bioportal.bioontology.org/ontologies/PO
po2name = {
    "PO:0025034": "Leaf",
    "PO:0020142": "Stem Internode",
    "PO:0009047": "Stem",
    "PO:0009005": "Root",
    "PO:0000055": "Bud",
    "PO:0004709": "Auxiliary Bud",
    "PO:0009046": "Flower",
    "PO:0009013": "Meristem",
    "PO:0020121": "Lateral Root",
    "PO:0020040": "Leaf Base",
    "PO:0020137": "Leaf Apex",
    "PO:0020141": "Stem Node",
    "PO:0025223": "Vegetative Shoot Apex",
    "PO:0025178": "Stem Epidermis",
    "PO:0009011": "Plant Structure",
    "PO:0006109": "Pith",
}
# Using Plant Trait Ontology (TO) for trait mapping
# This dictionary maps specific trait descriptions related to pathogen resistance/susceptibility, stress tolerance/susceptibility,
# and other characteristics to their corresponding terms in the Plant Trait Ontology (TO),
# https://bioportal.bioontology.org/ontologies/PTO
trait2TO = {
    # Fungal pathogen resistance/susceptibility
    "TO:0000439": "Smut Susceptible",
    # Viral pathogen resistance/susceptibility
    "TO:0000148": "Viral Disease Response",
    # Bacterial pathogen resistance/susceptibility
    "TO:0000315": "Bacterial Disease Response",
    # Drought stress tolerance/susceptibility
    "TO:0000276": "Drought Tolerant",
    "TO:0000188": "Drought Sensitive",
    # Insect pest resistance/susceptibility
    "TO:0000261": "Highly Susceptible to S. frugiperda",
    # Nematod resistance/susceptibility
    "TO:0000384": "Susceptibility to Pratylenchus zeae",
    # Temperature stress tolerance/susceptibility
    "TO:0000303": "Low temperature stress response",
    # Nitrogen requirement
    "TO:0000011": "Nitrogen sensitive",
    # Sucrose related traits
    "TO:0000328": "Sucrose content",
    # Chemical sensitivity
    "TO:0000482": "Chemical stress response",
    # Other traits that don't fit neatly into the above categories
    "TO:0000326": "Leaf color",
}

In [ ]:
ontology_map = {
    **time2UO,
    **time2PO,
    **peco2name,
    **po2name,
    **trait2TO,
}

def replace_ontology(value):
    if pd.isna(value):
        return value

    value = str(value).strip()

    # Exact match first
    if value in ontology_map:
        return ontology_map[value]

    # Handle comma-separated IDs
    if "," in value:
        return ", ".join(
            ontology_map.get(v.strip(), v.strip())
            for v in value.split(",")
        )

    return ontology_map.get(value, value)

# Only use to transform columns that contain ontology IDs, to avoid unintended replacements in other columns
ontology_cols = [col for col in df.columns if 'ontology' in col]
for col in ontology_cols:
    df[col] = df[col].apply(replace_ontology)

In [6]:
def create_edges_merge(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms.
    """
    edges = set()
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        # Self-join on each ontology column
        merged = df[['BioProject', col]].merge(
            df[['BioProject', col]], 
            on=col, 
            how='inner'
        )
        # Filter to keep only run1 < run2 to avoid duplicates
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        edges.update(zip(merged['BioProject_x'], merged['BioProject_y']))
    
    return list(edges)

In [7]:
def create_edges_merge_by_ontology_type(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms.
    """
    edges = {}  # (BioProject_x, BioProject_y) -> list of ontology cols
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        # Self-join on each ontology column
        merged = df[['BioProject', col]].merge(
            df[['BioProject', col]], 
            on=col, 
            how='inner'
        )
        # Filter to keep only run1 < run2 to avoid duplicates
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        
        for bp_x, bp_y in zip(merged['BioProject_x'], merged['BioProject_y']):
            key = (bp_x, bp_y)
            if key not in edges:
                edges[key] = set()
            edges[key].add(col)
    
    # Return list of (BioProject_x, BioProject_y, [ontologies])
    return [(a, b, sorted(cols)) for (a, b), cols in edges.items()]

In [13]:
def create_edges_merge_by_ontology_value(df: pd.DataFrame) -> list:
    """
    Create edges based on shared ontology terms using a self-join approach, grouping by ontology values.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects that share ontology terms, grouped by ontology values.
    """
    edges = {}
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        merged = df[['BioProject', col]].dropna(subset=[col]).merge(  # drop NaN before merging
            df[['BioProject', col]].dropna(subset=[col]), 
            on=col, 
            how='inner'
        )
        merged = merged[merged['BioProject_x'] < merged['BioProject_y']]
        
        for bp_x, bp_y, ontology_value in zip(merged['BioProject_x'], merged['BioProject_y'], merged[col]):
            key = (bp_x, bp_y)
            if key not in edges:
                edges[key] = set()
            edges[key].add(ontology_value)
    
    return [(a, b, sorted(cols)) for (a, b), cols in edges.items()]

In [8]:
def create_edges_merge_ontology2node(df: pd.DataFrame) -> set:
    """
    Create edges between BioProjects and ontology terms, treating ontologies as nodes.
    Args:
        df (pd.DataFrame): The input DataFrame containing BioProject and ontology columns.
    Returns:
        list: A list of tuples representing edges between BioProjects and ontology terms.
    """
    edges = set()
    ontology_cols = [col for col in df.columns if 'ontology' in col]
    
    for col in ontology_cols:
        merged = df[['BioProject', col]].dropna(subset=[col]).drop_duplicates()
        
        for bp, ont in zip(merged['BioProject'], merged[col]):
            for o in ont.split(','):
                edges.add((bp, o.strip()))

    return edges

In [9]:
edges = create_edges_merge_ontology2node(df) # or other functions created above
print(f"Number of edges: {len(edges)}")

Number of edges: 374


In [12]:
graph = igraph.Graph.TupleList(
    edges,
    directed=False,
    vertex_name_attr='BioProject',
    edge_attrs=['ontologies'] # just for the functions that return ontologies as edge attributes
)

In [13]:
graph.summary()

'IGRAPH U--- 196 374 -- \n+ attr: BioProject (v), ontologies (e)'

In [17]:
def create_ontology_table(edges, *ontology_dicts):
    # Merge ontology dictionaries
    ontology_map = {}
    ontology_source = {}

    for source_name, d in ontology_dicts:
        ontology_map.update(d)
        ontology_source.update({k: source_name for k in d})

    # Compute degrees
    degree = Counter(ont for _, ont in edges)

    # Build table
    return pd.DataFrame({
        "Ontology Term": list(ontology_map.keys()),
        "Name": list(ontology_map.values()),
        "Ontology": [ontology_source[k] for k in ontology_map],
        "Node Degree": [degree.get(k, 0) for k in ontology_map]
    }).sort_values("Node Degree", ascending=False)

In [19]:
ontology_df = create_ontology_table(
    edges,
    ("UO", time2UO),
    ("PO (stage)", time2PO),
    ("PECO", peco2name),
    ("PO (anatomy)", po2name),
    ("TO", trait2TO)
)

In [20]:
# Export the ontology table to a CSV file
ontology_df.to_csv('../raw/ontology_table.csv', index=False)

In [ ]:
graph.es['ontologies'] = [o for o in graph.es['ontologies']] # just for the functions that return ontologies as edge attributes

# Export graphml to use in gephi or cytoscape
graph.write_graphml('../raw/graph_sugarcane.graphml')